# Разбор Редкого Coarse-Хвоста Для Класса `O`

Цель:
- Воспроизводимо показать, где именно пропадает true `O` в текущем coarse pipeline.
- Отделить `quality_gate`-эффект от поведения coarse-model.
- Проверить, во что coarse-model переводит surviving `O` rows и насколько уверенно.
- Зафиксировать evidence до любых изменений модели, ребаланса или policy-правок.


In [1]:
# Setup: repo root, sys.path и базовые визуальные настройки.
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd
import seaborn as sns
from IPython.display import display


def find_repo_root(start_path: Path) -> Path:
    current_path = start_path.resolve()
    while current_path != current_path.parent:
        if (current_path / 'src').exists() and (current_path / 'analysis').exists():
            return current_path
        current_path = current_path.parent
    raise RuntimeError('Could not locate repository root.')


REPO_ROOT = find_repo_root(Path.cwd())
SRC_ROOT = REPO_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

sns.set_theme(style='whitegrid', context='talk')
pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 160)


In [2]:
# Импортируем review helpers после добавления src в sys.path.
from exohost.reporting.archive_research.coarse_o_review import (
    load_coarse_o_review_bundle_from_env,
    build_o_source_quality_summary_frame,
    build_o_source_quality_reason_frame,
    build_o_scored_prediction_frame,
    build_o_predicted_physics_summary_frame,
    build_o_high_confidence_non_o_preview_frame,
    build_o_final_outcome_distribution_frame,
    build_o_final_coarse_distribution_frame,
    build_o_final_reason_frame,
)


## План

1. Загружаем true `O` rows из `lab.gaia_mk_quality_gated`.
2. Сначала смотрим, сколько `O` теряется на `quality_gate`.
3. Потом считаем coarse predictions только на `quality_state = pass` части.
4. Отдельно смотрим физический профиль predicted groups.
5. После этого связываем те же true `O` rows с текущим `final decision` run.
6. В конце смотрим самые уверенные non-`O` кейсы и формулируем вывод.


In [3]:
# Конфигурация notebook.
COARSE_MODEL_RUN_DIR = REPO_ROOT / 'artifacts/models/gaia_id_coarse_classification__hist_gradient_boosting__2026_03_28_215003_509969'
FINAL_DECISION_RUN_DIR = REPO_ROOT / 'artifacts/decisions/hierarchical_final_decision_2026_03_29_111132_270743'
SOURCE_LIMIT = None
PREVIEW_ROWS = 20

if not COARSE_MODEL_RUN_DIR.exists():
    raise FileNotFoundError(COARSE_MODEL_RUN_DIR)
if not FINAL_DECISION_RUN_DIR.exists():
    raise FileNotFoundError(FINAL_DECISION_RUN_DIR)


In [4]:
# Загружаем `O` review bundle.
bundle = load_coarse_o_review_bundle_from_env(
    coarse_model_run_dir=COARSE_MODEL_RUN_DIR,
    final_decision_run_dir=FINAL_DECISION_RUN_DIR,
    source_limit=SOURCE_LIMIT,
    dotenv_path='.env',
)

pd.DataFrame(
    [
        {
            'n_rows_o_source': int(bundle.source_df.shape[0]),
            'n_rows_o_pass': int(bundle.pass_source_df.shape[0]),
            'n_rows_o_scored': int(bundle.scored_pass_df.shape[0]),
            'n_rows_o_final': int(bundle.final_o_df.shape[0]),
        }
    ]
)


,n_rows_o_source,n_rows_o_pass,n_rows_o_scored,n_rows_o_final
0,6372,1725,1725,6372


In [5]:
# Где true `O` теряется на quality-gate.
display(build_o_source_quality_summary_frame(bundle.source_df))
display(build_o_source_quality_reason_frame(bundle.source_df, top_n=10))


,quality_state,n_rows,share
0,unknown,2423,0.380257
1,reject,2224,0.349027
2,pass,1725,0.270716


,quality_reason,n_rows,share
0,missing_core_features,2224,0.349027
1,missing_radius_flame,2033,0.319052
2,pass,1725,0.270716
3,high_ruwe,284,0.044570
4,low_parallax_snr,106,0.016635


In [6]:
# Что coarse-model предсказывает на surviving `O` rows.
display(build_o_scored_prediction_frame(bundle.scored_pass_df))
display(build_o_predicted_physics_summary_frame(bundle.scored_pass_df))


,coarse_predicted_label,n_rows,share,mean_confidence,median_confidence,max_confidence
0,B,1189,0.689275,0.999744,0.999846,0.999863
1,F,353,0.204638,0.997912,0.999861,0.999861
2,G,79,0.045797,0.999861,0.999861,0.999861
3,A,75,0.043478,0.984721,0.999718,0.999851
4,K,28,0.016232,0.999853,0.999860,0.999868
5,M,1,0.000580,0.999861,0.999861,0.999861


,coarse_predicted_label,n_rows,median_teff_gspphot,median_logg_gspphot,median_bp_rp,median_radius_flame
0,B,1189,15610.3550,3.67260,0.477653,4.777008
1,F,353,6596.8325,3.49660,0.758043,3.308116
2,G,79,5783.0947,4.03530,0.926167,1.632382
3,A,75,8200.6660,3.03830,0.880737,5.579243
4,K,28,4817.8105,2.74335,1.506738,10.659700
5,M,1,3602.2760,0.57710,4.305430,17.189648


In [7]:
# Какой downstream outcome получают true `O` rows в current final run.
display(build_o_final_outcome_distribution_frame(bundle.final_o_df))
display(build_o_final_coarse_distribution_frame(bundle.final_o_df))
display(build_o_final_reason_frame(bundle.final_o_df, top_n=10))


,final_domain_state,n_rows,share
0,unknown,4647,0.729284
1,id,1614,0.253296
2,ood,111,0.017420


,final_coarse_class,n_rows,share
0,<NA>,4758,0.746704
1,B,1101,0.172787
2,F,350,0.054928
3,G,79,0.012398
4,A,55,0.008632
5,K,28,0.004394
6,M,1,0.000157


,final_decision_reason,n_rows,share
0,quality_unknown,2423,0.380257
1,quality_reject,2224,0.349027
2,refinement_accepted,1614,0.253296
3,hard_ood,111,0.017420


In [8]:
# Самые уверенные случаи, где true `O` ушел в другой coarse class.
display(build_o_high_confidence_non_o_preview_frame(bundle.scored_pass_df, top_n=PREVIEW_ROWS))


,source_id,coarse_predicted_label,coarse_probability_max,coarse_probability_margin,teff_gspphot,logg_gspphot,mh_gspphot,bp_rp,parallax,parallax_over_error,ruwe,radius_flame
0,3033461046693057536,K,0.999868,0.999840,4461.7856,1.9515,-0.3543,1.576572,0.403158,30.436184,1.120060,14.998648
1,2136007495388061184,K,0.999864,0.999834,4935.4736,3.0872,-0.0775,1.268957,0.769284,76.785576,1.074934,6.340798
2,6233150491818490880,B,0.999863,0.999836,17983.4800,3.8043,0.0482,0.090590,0.138179,6.129621,0.968518,5.596548
3,2910928898209142144,K,0.999863,0.999833,4732.2656,2.7676,-0.1966,1.214224,0.352053,35.328777,1.038548,8.391464
4,2067847193330730752,K,0.999863,0.999833,4664.2803,2.2997,-0.1528,1.506123,1.203325,114.453650,1.063489,11.763286
5,2067785070923661568,K,0.999863,0.999833,4539.4067,4.5718,0.0820,1.394720,1.846611,74.934235,1.039096,0.738404
6,5934699433615326336,K,0.999862,0.999832,4567.1000,2.0498,0.0209,1.664511,1.227685,93.293040,0.805493,24.108590
7,6726758537546311296,K,0.999862,0.999832,4825.4570,2.7487,0.1564,1.411803,0.558956,36.309284,0.775257,11.838723
8,5332734443894603136,K,0.999862,0.999832,4791.8174,2.6681,0.1184,1.453210,1.017925,85.387370,0.941556,10.731047
9,5893022514020520064,K,0.999862,0.999832,4619.7856,2.4641,0.1768,1.491728,1.220629,79.929740,0.970161,10.612527


## Опорные Источники

- [Gaia DR3 documentation index](https://gea.esac.esa.int/archive/documentation/GDR3/)
- [Gaia DR3 gaia_source semantics](https://gea.esac.esa.int/archive/documentation/GDR3/Gaia_archive/chap_datamodel/sec_dm_main_source_catalogue/ssec_dm_gaia_source.html)
- [Gaia DR3 astrophysical_parameters semantics](https://gea.esac.esa.int/archive/documentation/GDR3/Gaia_archive/chap_datamodel/sec_dm_astrophysical_parameter_tables/ssec_dm_astrophysical_parameters.html)
- [Gaia DR3 Apsis overview](https://gea.esac.esa.int/archive/documentation/GDR3/Data_analysis/chap_cu8par/sec_cu8par_intro/ssec_cu8par_intro_apsis.html)
- [scikit-learn: classification_report](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html)
- [scikit-learn: confusion_matrix](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html)
- [scikit-learn: compute_class_weight](https://scikit-learn.org/stable/modules/generated/sklearn.utils.class_weight.compute_class_weight.html)


## Следующие Шаги

- Если большая часть surviving true `O` физически уже не похожа на hot stars, сначала разбираем source/label consistency, а не лезем сразу в rebalance.
- Если true `O` rows физически горячие, но coarse-model все равно стабильно уводит их в `B/F`, тогда открываем отдельный coarse rare-tail improvement пакет.
- Threshold и priority здесь не трогаем: этот notebook отвечает только за rare-tail поведение coarse-stage.
